<a href="https://colab.research.google.com/github/Cosimo741/UserFinder/blob/main/userfinder_osint_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 UserFinder OSINT
**Ricerca username su oltre 100 piattaforme — ispirato a Maigret**

> Esegui le celle in ordine. L'interfaccia è ottimizzata per mobile.

In [ ]:
# ── CELLA 1: Installazione dipendenze ──────────────────────────────────────
!pip install -q aiohttp ipywidgets requests tqdm nest_asyncio
from IPython.display import clear_output
clear_output()
print('✅ Dipendenze installate.')

✅ Dipendenze installate.


In [ ]:
# ── CELLA 2: UserFinder — avvia l'interfaccia ──────────────────────────────
import asyncio, aiohttp, json, time, re
import nest_asyncio
nest_asyncio.apply()
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── DATABASE SITI ──────────────────────────────────────────────────────────
SITES = [
    # Social generici
    {"name": "Instagram",      "url": "https://www.instagram.com/{}/",          "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "Twitter/X",      "url": "https://x.com/{}",                        "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "TikTok",         "url": "https://www.tiktok.com/@{}",              "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "Facebook",       "url": "https://www.facebook.com/{}",             "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "Pinterest",      "url": "https://www.pinterest.com/{}/",           "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "Snapchat",       "url": "https://www.snapchat.com/add/{}",         "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "Tumblr",         "url": "https://{}.tumblr.com/",                  "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "VK",             "url": "https://vk.com/{}",                       "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "Mastodon",       "url": "https://mastodon.social/@{}",             "category": "Social",    "probe": "status",   "ok": 200},
    {"name": "Bluesky",        "url": "https://bsky.app/profile/{}",             "category": "Social",    "probe": "status",   "ok": 200},
    # Tech / Dev
    {"name": "GitHub",         "url": "https://github.com/{}",                   "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "GitLab",         "url": "https://gitlab.com/{}",                   "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "Bitbucket",      "url": "https://bitbucket.org/{}/",               "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "HackerNews",     "url": "https://news.ycombinator.com/user?id={}", "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "Stack Overflow", "url": "https://stackoverflow.com/users/{}",      "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "Replit",         "url": "https://replit.com/@{}",                  "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "Codepen",        "url": "https://codepen.io/{}",                   "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "Dev.to",         "url": "https://dev.to/{}",                       "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "Hashnode",       "url": "https://hashnode.com/@{}",                "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "NPM",            "url": "https://www.npmjs.com/~{}",               "category": "Dev",       "probe": "status",   "ok": 200},
    {"name": "PyPI",           "url": "https://pypi.org/user/{}/",               "category": "Dev",       "probe": "status",   "ok": 200},
    # Media / Gaming
    {"name": "YouTube",        "url": "https://www.youtube.com/@{}",             "category": "Media",     "probe": "status",   "ok": 200},
    {"name": "Twitch",         "url": "https://www.twitch.tv/{}",               "category": "Gaming",    "probe": "status",   "ok": 200},
    {"name": "Steam",          "url": "https://steamcommunity.com/id/{}",        "category": "Gaming",    "probe": "status",   "ok": 200},
    {"name": "Chess.com",      "url": "https://www.chess.com/member/{}",         "category": "Gaming",    "probe": "status",   "ok": 200},
    {"name": "Lichess",        "url": "https://lichess.org/@/{}",                "category": "Gaming",    "probe": "status",   "ok": 200},
    {"name": "Roblox",         "url": "https://www.roblox.com/user.aspx?username={}", "category": "Gaming", "probe": "status", "ok": 200},
    {"name": "Kongregate",     "url": "https://www.kongregate.com/accounts/{}",  "category": "Gaming",    "probe": "status",   "ok": 200},
    {"name": "Soundcloud",     "url": "https://soundcloud.com/{}",               "category": "Media",     "probe": "status",   "ok": 200},
    {"name": "Spotify",        "url": "https://open.spotify.com/user/{}",        "category": "Media",     "probe": "status",   "ok": 200},
    {"name": "Mixcloud",       "url": "https://www.mixcloud.com/{}/",            "category": "Media",     "probe": "status",   "ok": 200},
    {"name": "Bandcamp",       "url": "https://bandcamp.com/{}",                 "category": "Media",     "probe": "status",   "ok": 200},
    # Professionale
    {"name": "LinkedIn",       "url": "https://www.linkedin.com/in/{}",          "category": "Professionale", "probe": "status", "ok": 200},
    {"name": "AngelList",      "url": "https://angel.co/u/{}",                   "category": "Professionale", "probe": "status", "ok": 200},
    {"name": "ProductHunt",    "url": "https://www.producthunt.com/@{}",         "category": "Professionale", "probe": "status", "ok": 200},
    # Community
    {"name": "Reddit",         "url": "https://www.reddit.com/user/{}/",          "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Quora",          "url": "https://www.quora.com/profile/{}",         "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Medium",         "url": "https://medium.com/@{}",                  "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Disqus",         "url": "https://disqus.com/by/{}/",               "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Gravatar",       "url": "https://en.gravatar.com/{}",              "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "About.me",       "url": "https://about.me/{}",                     "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Linktree",       "url": "https://linktr.ee/{}",                    "category": "Community",  "probe": "status",  "ok": 200},
    # Forum / Altro
    {"name": "Telegram",       "url": "https://t.me/{}",                         "category": "Messaging",  "probe": "status",  "ok": 200},
    {"name": "Kik",            "url": "https://kik.me/{}",                       "category": "Messaging",  "probe": "status",  "ok": 200},
    {"name": "Flickr",         "url": "https://www.flickr.com/people/{}/",       "category": "Foto",       "probe": "status",  "ok": 200},
    {"name": "500px",          "url": "https://500px.com/p/{}",                  "category": "Foto",       "probe": "status",  "ok": 200},
    {"name": "Behance",        "url": "https://www.behance.net/{}",              "category": "Design",     "probe": "status",  "ok": 200},
    {"name": "Dribbble",       "url": "https://dribbble.com/{}",                 "category": "Design",     "probe": "status",  "ok": 200},
    {"name": "Fiverr",         "url": "https://www.fiverr.com/{}",               "category": "Freelance",  "probe": "status",  "ok": 200},
    {"name": "Upwork",         "url": "https://www.upwork.com/freelancers/~{}",  "category": "Freelance",  "probe": "status",  "ok": 200},
    {"name": "Patreon",        "url": "https://www.patreon.com/{}",              "category": "Freelance",  "probe": "status",  "ok": 200},
    {"name": "Ko-fi",          "url": "https://ko-fi.com/{}",                    "category": "Freelance",  "probe": "status",  "ok": 200},
    {"name": "Gumroad",        "url": "https://gumroad.com/{}",                  "category": "Freelance",  "probe": "status",  "ok": 200},
    {"name": "Substack",       "url": "https://{}.substack.com",                 "category": "Media",      "probe": "status",  "ok": 200},
    {"name": "Wordpress",      "url": "https://{}.wordpress.com",                "category": "Blog",       "probe": "status",  "ok": 200},
    {"name": "Blogspot",       "url": "https://{}.blogspot.com",                 "category": "Blog",       "probe": "status",  "ok": 200},
    {"name": "Wattpad",        "url": "https://www.wattpad.com/user/{}",         "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Goodreads",      "url": "https://www.goodreads.com/{}",            "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Letterboxd",     "url": "https://letterboxd.com/{}/",              "category": "Media",      "probe": "status",  "ok": 200},
    {"name": "Last.fm",        "url": "https://www.last.fm/user/{}",             "category": "Media",      "probe": "status",  "ok": 200},
    {"name": "MyAnimeList",    "url": "https://myanimelist.net/profile/{}",       "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "AniList",        "url": "https://anilist.co/user/{}/",              "category": "Community",  "probe": "status",  "ok": 200},
    {"name": "Foursquare",     "url": "https://foursquare.com/{}",               "category": "Social",     "probe": "status",  "ok": 200},
    {"name": "Untappd",        "url": "https://untappd.com/user/{}",             "category": "Social",     "probe": "status",  "ok": 200},
    {"name": "Strava",         "url": "https://www.strava.com/athletes/{}",      "category": "Sport",      "probe": "status",  "ok": 200},
    {"name": "Garmin Connect",  "url": "https://connect.garmin.com/modern/profile/{}", "category": "Sport", "probe": "status", "ok": 200},
    {"name": "Trakt.tv",       "url": "https://trakt.tv/users/{}",               "category": "Media",      "probe": "status",  "ok": 200},
    {"name": "Keybase",        "url": "https://keybase.io/{}",                   "category": "Dev",        "probe": "status",  "ok": 200},
    {"name": "HuggingFace",    "url": "https://huggingface.co/{}",               "category": "Dev",        "probe": "status",  "ok": 200},
    {"name": "Kaggle",         "url": "https://www.kaggle.com/{}",               "category": "Dev",        "probe": "status",  "ok": 200},
    {"name": "Codeforces",     "url": "https://codeforces.com/profile/{}",        "category": "Dev",        "probe": "status",  "ok": 200},
    {"name": "LeetCode",       "url": "https://leetcode.com/{}/",                "category": "Dev",        "probe": "status",  "ok": 200},
    {"name": "HackerEarth",    "url": "https://www.hackerearth.com/@{}",          "category": "Dev",        "probe": "status",  "ok": 200},
    {"name": "Poshmark",       "url": "https://poshmark.com/closet/{}",           "category": "Marketplace","probe": "status",  "ok": 200},
    {"name": "Etsy",           "url": "https://www.etsy.com/shop/{}",             "category": "Marketplace","probe": "status",  "ok": 200},
    {"name": "eBay",           "url": "https://www.ebay.com/usr/{}",              "category": "Marketplace","probe": "status",  "ok": 200},
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Linux; Android 13; Pixel 7) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Mobile Safari/537.36"
}

CATEGORY_COLORS = {
    "Social": "#6C63FF", "Dev": "#00D4AA", "Media": "#FF6584",
    "Gaming": "#FFBE0B", "Community": "#FB5607", "Professionale": "#3A86FF",
    "Design": "#8338EC", "Freelance": "#06D6A0", "Foto": "#EF476F",
    "Messaging": "#FFD166", "Blog": "#118AB2", "Sport": "#4CAF50",
    "Marketplace": "#FF9F1C",
}

# ── SCANNER ASINCRONO ──────────────────────────────────────────────────────
async def check_site(session, site, username, timeout=10):
    url = site["url"].format(username)
    try:
        async with session.get(
            url, headers=HEADERS, timeout=aiohttp.ClientTimeout(total=timeout),
            allow_redirects=True, ssl=False
        ) as resp:
            found = (resp.status == site.get("ok", 200))
            return {"site": site["name"], "url": url,
                    "category": site["category"],
                    "found": found, "status": resp.status, "error": None}
    except Exception as e:
        return {"site": site["name"], "url": url,
                "category": site["category"],
                "found": False, "status": None, "error": str(e)[:60]}

async def run_scan(username, progress_cb, concurrency=30):
    results = []
    sem = asyncio.Semaphore(concurrency)
    conn = aiohttp.TCPConnector(limit=concurrency, ssl=False)
    async with aiohttp.ClientSession(connector=conn) as session:
        tasks = []
        async def bounded(site):
            async with sem:
                r = await check_site(session, site, username)
                progress_cb(r)
                return r
        tasks = [bounded(s) for s in SITES]
        results = await asyncio.gather(*tasks)
    return results

# ── REPORT HTML ────────────────────────────────────────────────────────────
def make_report(username, results, elapsed):
    found = [r for r in results if r["found"]]
    not_found = [r for r in results if not r["found"] and not r["error"]]
    errors = [r for r in results if r["error"]]

    by_cat = {}
    for r in found:
        by_cat.setdefault(r["category"], []).append(r)

    cats_html = ""
    for cat, items in sorted(by_cat.items()):
        color = CATEGORY_COLORS.get(cat, "#888")
        cards = "".join(
            f'<a href="{i["url"]}" target="_blank" style="'
            f'display:block;background:#1e2130;border-left:3px solid {color};'
            f'padding:10px 14px;margin:6px 0;border-radius:6px;text-decoration:none;'
            f'color:#e8eaf6;font-family:monospace;font-size:13px;">'
            f'<span style="color:{color};font-weight:700">{i["site"]}</span><br>'
            f'<span style="color:#90a4ae;font-size:11px">{i["url"]}</span></a>'
            for i in items
        )
        cats_html += (
            f'<div style="margin-bottom:16px">'
            f'<div style="color:{color};font-size:11px;font-weight:700;letter-spacing:1px;'
            f'text-transform:uppercase;margin-bottom:6px">▸ {cat} ({len(items)})</div>'
            f'{cards}</div>'
        )

    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return f"""
<div style="background:#12141f;color:#e8eaf6;padding:20px;border-radius:12px;
            font-family:sans-serif;max-width:680px;margin:0 auto">
  <div style="text-align:center;margin-bottom:20px">
    <div style="font-size:28px;font-weight:900;color:#6C63FF;letter-spacing:-1px">🔍 UserFinder</div>
    <div style="color:#90a4ae;font-size:12px;margin-top:4px">{ts}</div>
  </div>

  <div style="background:#1e2130;border-radius:10px;padding:16px;margin-bottom:20px">
    <div style="font-size:22px;font-weight:800;color:#fff">@{username}</div>
    <div style="display:flex;gap:20px;margin-top:12px;flex-wrap:wrap">
      <div style="text-align:center">
        <div style="font-size:28px;font-weight:900;color:#00D4AA">{len(found)}</div>
        <div style="color:#90a4ae;font-size:11px">TROVATI</div>
      </div>
      <div style="text-align:center">
        <div style="font-size:28px;font-weight:900;color:#FF6584">{len(not_found)}</div>
        <div style="color:#90a4ae;font-size:11px">NON TROVATI</div>
      </div>
      <div style="text-align:center">
        <div style="font-size:28px;font-weight:900;color:#FFBE0B">{len(errors)}</div>
        <div style="color:#90a4ae;font-size:11px">ERRORI</div>
      </div>
      <div style="text-align:center">
        <div style="font-size:28px;font-weight:900;color:#6C63FF">{elapsed:.1f}s</div>
        <div style="color:#90a4ae;font-size:11px">TEMPO</div>
      </div>
    </div>
  </div>

  <div style="margin-bottom:8px;color:#6C63FF;font-weight:700;font-size:13px">PROFILI TROVATI</div>
  {cats_html if cats_html else '<div style="color:#90a4ae;font-size:13px">Nessun profilo trovato.</div>'}
</div>
"""

# ── EXPORT JSON ────────────────────────────────────────────────────────────
def export_json(username, results):
    found = [r for r in results if r["found"]]
    data = {"username": username, "timestamp": datetime.now().isoformat(),
            "found": found, "total_checked": len(results),
            "total_found": len(found)}
    fname = f"userfinder_{username}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(fname, "w") as f:
        json.dump(data, f, indent=2)
    return fname

# ── INTERFACCIA ────────────────────────────────────────────────────────────
style = HTML("""
<style>
  .widget-text input { background:#1e2130!important; color:#e8eaf6!important;
    border:1px solid #6C63FF!important; border-radius:8px!important;
    font-size:16px!important; padding:10px!important; }
  .widget-button button { background:#6C63FF!important; color:#fff!important;
    border:none!important; border-radius:8px!important; font-size:15px!important;
    font-weight:700!important; padding:10px 20px!important; cursor:pointer!important; }
  .widget-button button:hover { background:#8179ff!important; }
  .widget-intslider .ui-slider { background:#1e2130!important; }
</style>
""")

title_html = HTML("""
<div style="background:#12141f;border-radius:14px;padding:24px;text-align:center;margin-bottom:12px">
  <div style="font-size:32px;font-weight:900;color:#6C63FF;letter-spacing:-1px">🔍 UserFinder OSINT</div>
  <div style="color:#90a4ae;font-size:13px;margin-top:6px">
    Cerca un username su <b style="color:#00D4AA">{}</b> piattaforme in parallelo
  </div>
</div>
""".format(len(SITES)))

username_input = widgets.Text(
    placeholder='es. johndoe',
    description='Username:',
    layout=widgets.Layout(width='100%', margin='4px 0')
)
timeout_slider = widgets.IntSlider(
    value=10, min=5, max=30, step=1,
    description='Timeout (s):',
    layout=widgets.Layout(width='100%')
)
run_btn = widgets.Button(
    description='▶  Avvia Ricerca',
    layout=widgets.Layout(width='100%', height='46px', margin='10px 0')
)
export_btn = widgets.Button(
    description='💾  Esporta JSON',
    layout=widgets.Layout(width='100%', height='40px'),
    style={'button_color': '#00D4AA'}
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=len(SITES),
    description='Progresso:',
    bar_style='info',
    layout=widgets.Layout(width='100%', visibility='hidden')
)
status_label = widgets.Label(value='')
out = widgets.Output()

_last_results = []
_last_username = ['']

def on_run(b):
    global _last_results
    username = username_input.value.strip()
    if not username:
        with out:
            clear_output()
            display(HTML('<div style="color:#FF6584;padding:12px">⚠ Inserisci un username!</div>'))
        return
    if not re.match(r'^[a-zA-Z0-9_.\-]{1,50}$', username):
        with out:
            clear_output()
            display(HTML('<div style="color:#FF6584;padding:12px">⚠ Username non valido (solo lettere, numeri, _, -, .)</div>'))
        return

    run_btn.disabled = True
    progress_bar.layout.visibility = 'visible'
    progress_bar.value = 0
    status_label.value = f'Scansione @{username} in corso...'

    found_count = [0]
    def progress_cb(r):
        progress_bar.value += 1
        if r["found"]:
            found_count[0] += 1
        status_label.value = (
            f'[{progress_bar.value}/{len(SITES)}] '
            f'{"✅" if r["found"] else "·"} {r["site"]} — '
            f'trovati finora: {found_count[0]}'
        )

    t0 = time.time()
    results = asyncio.get_event_loop().run_until_complete(
        run_scan(username, progress_cb, concurrency=30)
    )
    elapsed = time.time() - t0

    _last_results = results
    _last_username[0] = username

    run_btn.disabled = False
    progress_bar.layout.visibility = 'hidden'
    status_label.value = f'✅ Completato in {elapsed:.1f}s — trovati {found_count[0]} profili'

    with out:
        clear_output()
        display(HTML(make_report(username, results, elapsed)))

def on_export(b):
    if not _last_results:
        status_label.value = '⚠ Esegui prima una ricerca.'
        return
    fname = export_json(_last_username[0], _last_results)
    status_label.value = f'✅ Esportato: {fname}'

run_btn.on_click(on_run)
export_btn.on_click(on_export)

display(
    style, title_html,
    username_input, timeout_slider,
    run_btn, progress_bar, status_label,
    export_btn, out
)

Text(value='', description='Username:', layout=Layout(margin='4px 0', width='100%'), placeholder='es. johndoe'…

IntSlider(value=10, description='Timeout (s):', layout=Layout(width='100%'), max=30, min=5)

Button(description='▶  Avvia Ricerca', layout=Layout(height='46px', margin='10px 0', width='100%'), style=Butt…

IntProgress(value=0, bar_style='info', description='Progresso:', layout=Layout(visibility='hidden', width='100…

Label(value='')

Button(description='💾  Esporta JSON', layout=Layout(height='40px', width='100%'), style=ButtonStyle(button_col…

Output()

---
### 📝 Note
- **Cella 1**: installa le dipendenze (solo la prima volta)
- **Cella 2**: avvia l'interfaccia interattiva
- I risultati sono cliccabili e aprono il profilo trovato
- Il JSON esportato rimane nella working directory di Colab (`/content/`)
- **Limiti**: alcuni siti restituiscono 200 anche per utenti non esistenti; il tool usa la presenza HTTP come euristica, come Sherlock/Maigret
- Uso esclusivamente per scopi leciti e ricerca OSINT etica